# 320 — Condition classification (audio / picture / reading)

Decodes the **stimulus condition** from a single electrode's spectro-temporal response.
3 classes, one sample per high-activity electrode × condition, run for **each of the 4
feature variants × each classifier** (logistic regression + random forest) = 8 experiments.
Compare the **matched pairs** `full_300 vs hg_300` and `full_30 vs hg_30` (see 390).

Every metric below comes from **nested GroupKFold by patient** — the outer fold holds out
whole patients for testing, the inner fold tunes hyper-parameters, and no patient ever
appears in both train and test. That makes the numbers an estimate of *generalisation to a
new patient*, not memorisation of these ones.

**How to read each run** (full narrative + figures in `390_results.ipynb`):
- **Balanced accuracy vs chance (0.333)** — headline separability, robust to imbalance.
- **Permutation p** — is balanced accuracy above a patient-shuffled-label null?
- **Confusion matrix** — which conditions get mixed up with which.
- **Per-class strength** — recall ± bootstrap CI, with FDR significance stars.
- **Feature importance** — which bands / time bins drove the separation.


In [1]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_classify as C

# ---------------- config knobs ----------------
CLASSIFIERS  = ('logreg', 'rf')
OUTER_SPLITS = 5      # outer GroupKFold (held-out test patients)
INNER_SPLITS = 3      # inner GroupKFold (hyper-parameter tuning)
N_PERM       = 200    # label-permutation null reps (set 0 to skip; rf is the slow part)
N_BOOT       = 1000   # bootstrap reps for per-class CIs (cheap)
RANDOM_STATE = 42
# All 8 variants. full_300 is the heavy one; subset for a fast first pass, e.g.
#   VARIANTS_TO_RUN = ('full_300', 'full_300_rn', 'm101_300', 'hg_300')
VARIANTS_TO_RUN = C.VARIANTS
print('classifiers:', CLASSIFIERS, '| variants:', VARIANTS_TO_RUN, '| n_perm:', N_PERM)


classifiers: ('logreg', 'rf') | variants: ('full_300', 'hg_300', 'full_300_rn', 'hg_300_rn', 'full_30', 'hg_30', 'full_30_rn', 'hg_30_rn') | n_perm: 200


## Run all condition experiments
8 variants × 2 classifiers = 16 runs (fewer if you subset `VARIANTS_TO_RUN`), each saved
under `outputs/classification/condition/<variant>/<classifier>/runs/<id>/`.


In [ ]:
# Full-spectrum (15 bands x 300 time) drives the per-class ERSP *response profile*
# shown for every run, regardless of which variant the classifier used — so HG runs
# still show the full spectrogram they are a slice of. n_cond=1 (class == condition).
Xf, yf, gf, mf, cf = C.load_arrays('condition', None, 'full_300')
profile = {'X': Xf, 'n_time': 300, 'n_cond': 1}

manifests = []
for v in VARIANTS_TO_RUN:
    X, y, groups, meta, cols = C.load_arrays('condition', None, v)
    for clf in CLASSIFIERS:
        m = C.run_experiment('condition', v, clf, X, y, groups, cols, meta,
                             profile=profile,
                             outer_splits=OUTER_SPLITS, inner_splits=INNER_SPLITS,
                             n_perm=N_PERM, n_boot=N_BOOT, random_state=RANDOM_STATE)
        manifests.append(m)
print('\ndone:', len(manifests), 'runs')



=== condition | full_300 | logreg -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\03_FBM_Classifying\outputs\classification\condition\full_300\logreg\runs\20260719_160521 ===
  outer fold 1/5: best={'clf__C': 0.01}
  outer fold 2/5: best={'clf__C': 10.0}
  outer fold 3/5: best={'clf__C': 10.0}
  outer fold 4/5: best={'clf__C': 10.0}
  outer fold 5/5: best={'clf__C': 10.0}


## Summary table


In [ ]:
df = C.list_runs()
df = df[df.task == 'condition']
df[['variant', 'classifier', 'balanced_accuracy', 'chance_level',
    'macro_f1', 'permutation_p']].round(4)
